# Busca Exaustiva A100: Funções RS Bent de Grau <= 3 em n=12

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/contatofilipevono-rgb/hrsbf-bent-verification/blob/colab-a100-2026-10-06/RUN_A100_N12_EXHAUSTIVE.ipynb)

Este notebook executa o controle exaustivo em GPU (A100 recomendada) para dimensionamento **$n=12$** ($2^{25} = 33.554.432$ funções candidatas).

### Objetivos científicos:
1. **Regra Antipodal em $n=12$**: Confirmar exaustivamente que **100% das funções bent RS** de grau $\le 3$ contêm a órbita quadrática antipodal $P_{12}$.
2. **Conjectura Cúbica Homogênea**: Confirmar que **zero funções cúbicas homogêneas** são bent em $n=12$.
3. **Validação do Lema de Dobramento (Folding)**: Testar o menor caso não-potência de 2 composto ($12 = 4 \times 3$) que exercita simultaneamente a redução por ordem ímpar ($m=3$) e a potência de 2 ($t=4$).

## 1. Clonar Repositório e Verificar GPU

In [ ]:
import os, subprocess, pathlib

REPO = 'https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git'
BRANCH = 'colab-a100-2026-10-06'
ROOT = pathlib.Path('/content/hrsbf-bent-verification')

if not (ROOT / '.git').exists():
    print('Clonando repositório...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(ROOT)], check=True)
else:
    print('Atualizando repositório...')
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'pull', 'origin', BRANCH], check=True)

os.chdir(ROOT)
print(f'Diretório de trabalho: {os.getcwd()}')

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python pesquisa_t32/gpu_probe.py

## 2. Testes Rápidos de Sanidade em GPU ($n=4$ e $n=8$)
Rodam em menos de 2 segundos para validar o backend CUDA contra os valores teóricos.

In [ ]:
print('--- Teste n=4 (esperado: 2 bent afins) ---')
!python rs_bent_exhaustive.py 4 12 --backend cuda

print('\n--- Teste n=8 (esperado: 64 bent afins) ---')
!python rs_bent_exhaustive.py 8 12 --backend cuda

## 3. Execução Exaustiva em $n=12$ ($33.554.432$ funções)
Na GPU A100, esta varredura leva tipicamente entre **30 a 60 minutos** (batch size $2^{12} = 4096$, com checkpointing automático a cada 64 lotes).

In [ ]:
import os
os.makedirs('results/rs_bent_n12/rev_2', exist_ok=True)

!python rs_bent_exhaustive.py 12 12 --backend cuda --resume --output-dir results/rs_bent_n12/rev_2

## 4. Visualizar Relatório e Validar a Regra Antipodal

In [ ]:
import json, pathlib

res_path = pathlib.Path('results/rs_bent_n12/rev_2/result_n12.json')
if res_path.exists():
    res = json.loads(res_path.read_text())
    print('================ RELATÓRIO FINAL n=12 ================\n')
    print(f"Candidatos testados:              {res['tested']} (esperado: {res['expected']})")
    print(f"Sobreviventes ao filtro D_J:      {res['passed_DJ_filter']}")
    print(f"Representantes Bent encontrados:  {res['bent_affine_representatives']}")
    print(f"Total Bent (com afins RS):        {res['bent_including_RS_affine']}")
    print(f"Bent SEM órbita antipodal P_12:   {res['bent_without_Pn']}  <-- DEVE SER ZERO")
    print(f"Cúbicas Homogêneas Bent:          {res['homogeneous_cubic_bent']}  <-- DEVE SER ZERO")
    print(f"Status:                           {res['status']}")
    print(f"Tempo de execução:                {res['elapsed_seconds']:.1f}s ({res['elapsed_seconds']/60:.2f} min)")
    print('\n======================================================')
    
    if res['bent_without_Pn'] == 0 and res['homogeneous_cubic_bent'] == 0 and res['complete']:
        print('✅ REGRA ANTIPODAL E INEXISTÊNCIA CÚBICA HOMOGÊNEA CONFIRMADAS EM n=12!')
    else:
        print('⚠️ ATENÇÃO: Verifique os números acima!')
else:
    print('Arquivo de resultados ainda não gerado.')

## 5. Download dos Artefatos Certificados (Opcional)
Gera um `.zip` com os arquivos `result_n12.json` e `bent_ids_n12.npy` para incorporar ao repositório local e ao artigo.

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive('rs_bent_n12_results', 'zip', 'results/rs_bent_n12/rev_2')
    files.download('rs_bent_n12_results.zip')
    print('Download iniciado!')
except Exception as e:
    print('Se não estiver no Google Colab, acesse os arquivos diretamente em results/rs_bent_n12/rev_2/')
